# 02 - E4 Preprocesamiento y normalización del dataset SPIDER

**Objetivo:** preparar el dataset SPIDER para una etapa posterior de baseline de segmentación sagital, sin entrenar modelos todavía.

Este notebook cubre:

- montaje de Google Drive y definición de rutas;
- detección y emparejamiento de imágenes y máscaras `.mha`;
- inspección de consistencia geométrica y estadística;
- normalización robusta por percentiles;
- validación visual de orientación/eje de trabajo;
- exportación de evidencias reproducibles para cerrar E4.

**Alcance:** no se entrena ningún modelo, no se ejecuta nnU-Net y no se guarda todavía todo el dataset preprocesado.

## 1. Instalación e importación de dependencias

In [ ]:
!pip -q install SimpleITK scikit-image

In [ ]:
from pathlib import Path
import json
import warnings

import SimpleITK as sitk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage import exposure

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["image.cmap"] = "gray"

print("SimpleITK:", sitk.Version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

## 2. Montaje de Google Drive y definición de rutas

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DATASET_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/SPIDER")
IMAGES_ROOT = DATASET_ROOT / "images"
MASKS_ROOT = DATASET_ROOT / "masks"

RESULTS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E4_preprocesamiento")
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

for path in [RESULTS_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("DATASET_ROOT:", DATASET_ROOT)
print("IMAGES_ROOT:", IMAGES_ROOT)
print("MASKS_ROOT:", MASKS_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("FIGURES_ROOT:", FIGURES_ROOT)
print("DOCS_ROOT:", DOCS_ROOT)

## 3. Detección de imágenes y máscaras

In [ ]:
def find_mha_files(root: Path):
    """Busca recursivamente archivos .mha y devuelve una lista ordenada."""
    return sorted(root.rglob("*.mha"))


image_paths = find_mha_files(IMAGES_ROOT)
mask_paths = find_mha_files(MASKS_ROOT)

print(f"Imágenes detectadas: {len(image_paths)}")
print(f"Máscaras detectadas: {len(mask_paths)}")

image_by_stem = {p.stem: p for p in image_paths}
mask_by_stem = {p.stem: p for p in mask_paths}

common_stems = sorted(set(image_by_stem) & set(mask_by_stem))
missing_masks = sorted(set(image_by_stem) - set(mask_by_stem))
missing_images = sorted(set(mask_by_stem) - set(image_by_stem))

pairs = [(stem, image_by_stem[stem], mask_by_stem[stem]) for stem in common_stems]

print(f"Pares imagen/máscara: {len(pairs)}")
print(f"Imágenes sin máscara: {len(missing_masks)}")
print(f"Máscaras sin imagen: {len(missing_images)}")

if missing_masks:
    print("Primeras imágenes sin máscara:", missing_masks[:10])
if missing_images:
    print("Primeras máscaras sin imagen:", missing_images[:10])

In [ ]:
pairing_report = pd.DataFrame({
    "category": ["images", "masks", "paired_cases", "images_without_mask", "masks_without_image"],
    "count": [len(image_paths), len(mask_paths), len(pairs), len(missing_masks), len(missing_images)],
})

pairing_report_path = RESULTS_ROOT / "spider_E4_pairing_report.csv"
pairing_report.to_csv(pairing_report_path, index=False)

pairing_report

## 4. Inspección de consistencia en múltiples casos

In [ ]:
def read_mha(path: Path):
    image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(image)  # orden NumPy: z, y, x
    return image, array


def unique_labels_summary(mask_array, max_labels=20):
    labels = np.unique(mask_array)
    labels_list = labels.tolist()
    if len(labels_list) > max_labels:
        return json.dumps(labels_list[:max_labels] + ["..."])
    return json.dumps(labels_list)


def inspect_pair(stem: str, image_path: Path, mask_path: Path):
    itk_img, img = read_mha(image_path)
    itk_mask, mask = read_mha(mask_path)

    return {
        "case_id": stem,
        "image_path": str(image_path),
        "mask_path": str(mask_path),
        "image_shape_zyx": tuple(img.shape),
        "mask_shape_zyx": tuple(mask.shape),
        "same_shape": tuple(img.shape) == tuple(mask.shape),
        "image_spacing_xyz": tuple(float(x) for x in itk_img.GetSpacing()),
        "mask_spacing_xyz": tuple(float(x) for x in itk_mask.GetSpacing()),
        "same_spacing": np.allclose(itk_img.GetSpacing(), itk_mask.GetSpacing()),
        "image_origin_xyz": tuple(float(x) for x in itk_img.GetOrigin()),
        "mask_origin_xyz": tuple(float(x) for x in itk_mask.GetOrigin()),
        "same_origin": np.allclose(itk_img.GetOrigin(), itk_mask.GetOrigin()),
        "image_direction": tuple(float(x) for x in itk_img.GetDirection()),
        "mask_direction": tuple(float(x) for x in itk_mask.GetDirection()),
        "same_direction": np.allclose(itk_img.GetDirection(), itk_mask.GetDirection()),
        "image_dtype": str(img.dtype),
        "mask_dtype": str(mask.dtype),
        "image_min": float(np.min(img)),
        "image_max": float(np.max(img)),
        "image_mean": float(np.mean(img)),
        "mask_min": float(np.min(mask)),
        "mask_max": float(np.max(mask)),
        "mask_mean": float(np.mean(mask)),
        "mask_unique_labels": unique_labels_summary(mask),
        "mask_nonzero_voxels": int(np.count_nonzero(mask)),
    }


rows = []
for stem, image_path, mask_path in pairs:
    try:
        rows.append(inspect_pair(stem, image_path, mask_path))
    except Exception as exc:
        rows.append({
            "case_id": stem,
            "image_path": str(image_path),
            "mask_path": str(mask_path),
            "read_error": repr(exc),
        })

consistency_df = pd.DataFrame(rows)
consistency_csv_path = RESULTS_ROOT / "spider_E4_consistency_summary.csv"
consistency_df.to_csv(consistency_csv_path, index=False)

print("CSV de consistencia:", consistency_csv_path)
consistency_df.head()

In [ ]:
if len(consistency_df) > 0:
    consistency_checks = {
        "total_pairs": int(len(consistency_df)),
        "same_shape_cases": int(consistency_df.get("same_shape", pd.Series(dtype=bool)).fillna(False).sum()),
        "same_spacing_cases": int(consistency_df.get("same_spacing", pd.Series(dtype=bool)).fillna(False).sum()),
        "same_origin_cases": int(consistency_df.get("same_origin", pd.Series(dtype=bool)).fillna(False).sum()),
        "same_direction_cases": int(consistency_df.get("same_direction", pd.Series(dtype=bool)).fillna(False).sum()),
        "read_error_cases": int(consistency_df.get("read_error", pd.Series(dtype=object)).notna().sum()) if "read_error" in consistency_df else 0,
    }
    display(pd.DataFrame([consistency_checks]))

    inconsistent = consistency_df[
        (consistency_df.get("same_shape", True) != True)
        | (consistency_df.get("same_spacing", True) != True)
        | (consistency_df.get("same_origin", True) != True)
        | (consistency_df.get("same_direction", True) != True)
    ]
    print("Casos con alguna inconsistencia geométrica:", len(inconsistent))
    display(inconsistent.head(20))
else:
    raise RuntimeError("No se encontraron pares imagen/máscara para inspeccionar.")

## 5. Normalización robusta de intensidades

In [ ]:
def robust_percentile_normalize(image_array, p_low=1, p_high=99, eps=1e-8, clip=True):
    """Normaliza una imagen a [0, 1] usando percentiles robustos.

    La máscara no debe pasar por esta función; las etiquetas deben conservarse intactas.
    """
    image_float = image_array.astype(np.float32)
    low, high = np.percentile(image_float, [p_low, p_high])

    if np.isclose(high, low):
        normalized = np.zeros_like(image_float, dtype=np.float32)
    else:
        normalized = (image_float - low) / (high - low + eps)

    if clip:
        normalized = np.clip(normalized, 0.0, 1.0)

    return normalized.astype(np.float32), {"p_low": float(low), "p_high": float(high)}


example_stem, example_image_path, example_mask_path = pairs[0]
example_itk, example_image = read_mha(example_image_path)
example_mask_itk, example_mask = read_mha(example_mask_path)
example_norm, norm_stats = robust_percentile_normalize(example_image, p_low=1, p_high=99)

print("Caso de ejemplo:", example_stem)
print("Imagen original:", example_image.shape, example_image.dtype, float(example_image.min()), float(example_image.max()))
print("Imagen normalizada:", example_norm.shape, example_norm.dtype, float(example_norm.min()), float(example_norm.max()))
print("Máscara intacta:", np.array_equal(example_mask, example_mask.copy()), example_mask.dtype, np.unique(example_mask)[:20])
print("Percentiles usados:", norm_stats)

In [ ]:
def representative_slice_index(mask_array=None, image_array=None, axis=2):
    """Devuelve un índice representativo priorizando el centro de masa de la máscara."""
    reference = mask_array if mask_array is not None and np.count_nonzero(mask_array) > 0 else image_array
    if reference is None:
        raise ValueError("Se requiere mask_array o image_array.")

    if mask_array is not None and np.count_nonzero(mask_array) > 0:
        projection = np.sum(mask_array > 0, axis=tuple(ax for ax in range(mask_array.ndim) if ax != axis))
        return int(np.argmax(projection))

    return int(reference.shape[axis] // 2)


def take_slice(array, axis, index):
    return np.take(array, indices=index, axis=axis)


SAGITTAL_AXIS = 2
example_slice_idx = representative_slice_index(example_mask, example_image, axis=SAGITTAL_AXIS)

before_slice = take_slice(example_image, SAGITTAL_AXIS, example_slice_idx)
after_slice = take_slice(example_norm, SAGITTAL_AXIS, example_slice_idx)
mask_slice = take_slice(example_mask, SAGITTAL_AXIS, example_slice_idx)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(before_slice, cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(after_slice, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Normalizada p1-p99")
axes[2].hist(example_image.ravel(), bins=80, alpha=0.55, label="Original")
axes[2].hist(example_norm.ravel(), bins=80, alpha=0.55, label="Normalizada")
axes[2].set_title("Distribución de intensidades")
axes[2].legend()

for ax in axes[:2]:
    ax.axis("off")

fig.suptitle(f"Comparación de normalización - {example_stem} - eje {SAGITTAL_AXIS}, slice {example_slice_idx}")
fig.tight_layout()

normalization_fig_path = FIGURES_ROOT / f"E4_normalizacion_antes_despues_{example_stem}.png"
fig.savefig(normalization_fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("Figura exportada:", normalization_fig_path)

## 6. Orientación y eje de trabajo

In [ ]:
def plot_multi_axis(image_array, mask_array=None, case_id="case", slice_strategy="mask"):
    """Visualiza cortes representativos en los tres ejes del arreglo z, y, x."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    chosen = {}

    for axis in range(3):
        if slice_strategy == "mask":
            idx = representative_slice_index(mask_array, image_array, axis=axis)
        else:
            idx = image_array.shape[axis] // 2
        chosen[axis] = idx

        img_slice = take_slice(image_array, axis, idx)
        axes[axis].imshow(img_slice, cmap="gray")

        if mask_array is not None:
            msk_slice = take_slice(mask_array, axis, idx)
            axes[axis].imshow(np.ma.masked_where(msk_slice == 0, msk_slice), cmap="autumn", alpha=0.45)

        axes[axis].set_title(f"Eje {axis} - slice {idx}")
        axes[axis].axis("off")

    fig.suptitle(f"Inspección multi-eje - {case_id}")
    fig.tight_layout()
    return fig, chosen


fig, chosen_slices = plot_multi_axis(example_norm, example_mask, case_id=example_stem)
multi_axis_fig_path = FIGURES_ROOT / f"E4_multieje_normalizado_{example_stem}.png"
fig.savefig(multi_axis_fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("Slices elegidos por eje:", chosen_slices)
print("Figura exportada:", multi_axis_fig_path)
print("Eje de trabajo propuesto para inspección sagital:", SAGITTAL_AXIS)
print("Nota: la elección del eje 2 se confirma visualmente en este caso y debe verificarse en más casos antes del baseline final.")

In [ ]:
def plot_overlay(image_array, mask_array, axis=2, slice_idx=None, case_id="case"):
    if slice_idx is None:
        slice_idx = representative_slice_index(mask_array, image_array, axis=axis)

    img_slice = take_slice(image_array, axis, slice_idx)
    mask_slice = take_slice(mask_array, axis, slice_idx)

    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    ax.imshow(img_slice, cmap="gray", vmin=0, vmax=1)
    ax.imshow(np.ma.masked_where(mask_slice == 0, mask_slice), cmap="autumn", alpha=0.45)
    ax.set_title(f"Overlay normalizado + máscara - {case_id} - eje {axis}, slice {slice_idx}")
    ax.axis("off")
    fig.tight_layout()
    return fig, slice_idx


fig, overlay_slice_idx = plot_overlay(example_norm, example_mask, axis=SAGITTAL_AXIS, case_id=example_stem)
overlay_fig_path = FIGURES_ROOT / f"E4_overlay_normalizado_mascara_{example_stem}.png"
fig.savefig(overlay_fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("Figura exportada:", overlay_fig_path)

## 7. Preparación para baseline sagital

In [ ]:
def prepare_case_for_sagittal_baseline(image_path: Path, mask_path: Path, axis=2, p_low=1, p_high=99):
    """Carga un caso, normaliza la imagen y devuelve datos mínimos para validar el flujo.

    No altera la máscara y no guarda el dataset completo preprocesado.
    """
    itk_img, image = read_mha(image_path)
    itk_mask, mask = read_mha(mask_path)

    if image.shape != mask.shape:
        raise ValueError(f"Shape incompatible: image={image.shape}, mask={mask.shape}")

    normalized, stats = robust_percentile_normalize(image, p_low=p_low, p_high=p_high)
    slice_idx = representative_slice_index(mask, normalized, axis=axis)

    metadata = {
        "case_id": image_path.stem,
        "image_path": str(image_path),
        "mask_path": str(mask_path),
        "shape_zyx": tuple(int(x) for x in image.shape),
        "spacing_xyz": tuple(float(x) for x in itk_img.GetSpacing()),
        "origin_xyz": tuple(float(x) for x in itk_img.GetOrigin()),
        "direction": tuple(float(x) for x in itk_img.GetDirection()),
        "axis": int(axis),
        "representative_slice_idx": int(slice_idx),
        "normalization": {
            "method": "percentile",
            "p_low": float(p_low),
            "p_high": float(p_high),
            "value_low": stats["p_low"],
            "value_high": stats["p_high"],
        },
    }

    return {
        "image_normalized": normalized,
        "mask": mask,
        "metadata": metadata,
        "sagittal_image_slice": take_slice(normalized, axis, slice_idx),
        "sagittal_mask_slice": take_slice(mask, axis, slice_idx),
    }


prepared = prepare_case_for_sagittal_baseline(example_image_path, example_mask_path, axis=SAGITTAL_AXIS)

print("Caso preparado:", prepared["metadata"]["case_id"])
print("Imagen normalizada:", prepared["image_normalized"].shape, prepared["image_normalized"].dtype)
print("Máscara:", prepared["mask"].shape, prepared["mask"].dtype)
print("Slice sagital:", prepared["sagittal_image_slice"].shape)
prepared["metadata"]

In [ ]:
npz_path = RESULTS_ROOT / f"E4_preprocessed_example_{prepared['metadata']['case_id']}.npz"

np.savez_compressed(
    npz_path,
    image_normalized=prepared["image_normalized"],
    mask=prepared["mask"],
    sagittal_image_slice=prepared["sagittal_image_slice"],
    sagittal_mask_slice=prepared["sagittal_mask_slice"],
    metadata=json.dumps(prepared["metadata"]),
)

print("Ejemplo .npz exportado:", npz_path)

## 8. Conclusión técnica y evidencias

In [ ]:
conclusion_md = f"""# Conclusión técnica - E4 Preprocesamiento y normalización SPIDER

## Objetivo

Se preparó el dataset SPIDER para una futura etapa de baseline de segmentación sagital, sin entrenamiento y sin ejecutar nnU-Net.

## Resultados del notebook

- Imágenes `.mha` detectadas: {len(image_paths)}
- Máscaras `.mha` detectadas: {len(mask_paths)}
- Pares imagen/máscara construidos por nombre de archivo: {len(pairs)}
- Imágenes sin máscara: {len(missing_masks)}
- Máscaras sin imagen: {len(missing_images)}
- Caso de ejemplo validado: `{example_stem}`
- Eje de trabajo propuesto para inspección sagital: eje {SAGITTAL_AXIS}

## Normalización

Se implementó normalización robusta por percentiles p1-p99 sobre la imagen. La máscara no se normaliza ni se modifica, para conservar sus etiquetas originales.

## Evidencias exportadas

- CSV de emparejamiento: `{pairing_report_path}`
- CSV de consistencia: `{consistency_csv_path}`
- PNG antes/después de normalización: `{normalization_fig_path}`
- PNG multi-eje normalizado: `{multi_axis_fig_path}`
- PNG overlay normalizado + máscara: `{overlay_fig_path}`
- Ejemplo `.npz` de flujo preprocesado: `{npz_path}`

## Nota sobre orientación

La exploración previa y esta validación visual sostienen que el eje 2 es el más representativo para inspección sagital en el caso de ejemplo. Esta decisión debe verificarse en más casos antes de fijarla definitivamente para el baseline.

## Criterio de aceptación

El flujo permite emparejar imágenes y máscaras, detectar inconsistencias, normalizar una imagen sin alterar la máscara, seleccionar un slice representativo y exportar evidencia reproducible para preparar el baseline sagital.
"""

conclusion_path = DOCS_ROOT / "E4_preprocesamiento_normalizacion_conclusion.md"
conclusion_path.write_text(conclusion_md, encoding="utf-8")

print(conclusion_md)
print("Conclusión Markdown exportada:", conclusion_path)

## 9. Cierre de E4

Si las celdas anteriores se ejecutan sin errores, el notebook cumple el criterio de aceptación definido para esta etapa:

- empareja imágenes y máscaras;
- detecta inconsistencias geométricas o de lectura;
- normaliza intensidades sin alterar la máscara;
- selecciona un slice representativo en eje sagital;
- exporta evidencia reproducible para preparar el baseline.

**Próximo paso sugerido:** revisar más casos con la visualización multi-eje antes de congelar definitivamente el eje sagital para el baseline. No avanzar todavía a entrenamiento.